# 3D-ST — Plantilla de Google Colab

Esta plantilla general permite cargar una foto y generar un modelo 3D con TripoSR. Si llegaste desde la web de 3D-ST y descargaste el cuaderno personalizado, la foto ya está incorporada y no necesitas cargarla otra vez.

## 1. Seleccionar fotografía

Esta celda es para usar la plantilla manualmente. En el cuaderno personalizado generado por la web de 3D-ST, puedes omitirla porque la imagen ya viene incluida.

In [ ]:
from google.colab import files
from pathlib import Path

cargas = files.upload()
if not cargas:
    raise RuntimeError('Carga una fotografía PNG, JPG, JPEG o WebP.')
nombre = next(iter(cargas))
imagen = Path('/content') / nombre
print('Fotografía seleccionada:', imagen)

## 2. Preparar entorno

Si está disponible, activa una GPU desde Entorno de ejecución → Cambiar tipo de entorno de ejecución. La instalación puede tardar varios minutos.

In [ ]:
import os, pathlib, subprocess, sys
def ejecutar(cmd):
    print('$', ' '.join(cmd))
    r = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.returncode != 0:
        print(r.stdout[-6000:])  # mostrar el error real de pip
        raise RuntimeError('Falló: ' + ' '.join(str(c) for c in cmd[:6]) + ' ... Revisa el registro de arriba.')

# Colab administra su propio Python (en esta sesión puede ser 3.13).
# No creamos un entorno virtual, porque algunas imágenes de Colab no incluyen
# ensurepip/venv. Se conserva el PyTorch y CUDA que Colab ya tiene instalados.
PY = sys.executable
print('Python de Colab:', sys.version)
try:
    import torch
    print('PyTorch preinstalado:', torch.__version__)
    print('CUDA disponible:', torch.cuda.is_available())
except Exception as e:
    raise RuntimeError('No se pudo importar PyTorch del entorno de Colab: ' + str(e))

ejecutar([PY, '-m', 'pip', 'install', '--upgrade', 'scikit-build-core', 'pybind11', 'ninja', 'cmake'])
ejecutar([PY, '-m', 'pip', 'install', '--no-cache-dir', '--no-build-isolation', 'git+https://github.com/tatsy/torchmcubes.git'])
ejecutar([PY, '-m', 'pip', 'install', 'omegaconf==2.3.0', 'Pillow', 'einops', 'transformers>=4.40,<5', 'trimesh', 'rembg', 'onnxruntime', 'huggingface-hub', 'imageio[ffmpeg]', 'gradio', 'xatlas', 'moderngl'])
if not pathlib.Path('/content/TripoSR/run.py').exists():
    ejecutar(['git', 'clone', 'https://github.com/VAST-AI-Research/TripoSR.git', '/content/TripoSR'])
ejecutar([PY, '-c', "import torch, torchmcubes, rembg, onnxruntime; print('PyTorch:', torch.__version__); print('CUDA:', torch.version.cuda); print('GPU disponible:', torch.cuda.is_available()); print('Dependencias: OK')"])
try:
    subprocess.run(['nvidia-smi'], check=True)
except Exception:
    print('No se detectó una GPU NVIDIA. Si Colab ofrece GPU, actívala en Entorno de ejecución → Cambiar tipo de entorno de ejecución.')
print('Entorno preparado.')


## 3. Generar el modelo 3D

TripoSR estima las partes no visibles. El resultado puede necesitar reparación antes de imprimirlo en 3D.

In [ ]:
import pathlib, subprocess, os
import sys
PY = sys.executable
SALIDA = '/content/3dst-output'
os.makedirs(SALIDA, exist_ok=True)
import torch
hay_gpu = torch.cuda.is_available()
print('GPU disponible:', hay_gpu)
if not hay_gpu:
    print('AVISO: sin GPU el proceso es lento y usa mucha RAM. Activa una GPU en Entorno de ejecución → Cambiar tipo de entorno de ejecución.')
# Resolución de la malla: más alta = más detalle, pero más memoria. Si se agota, baja sola.
resoluciones = [320, 256, 192] if hay_gpu else [192, 128]
print('Generando el modelo 3D...')
for res in resoluciones:
    print('Intentando con mc-resolution =', res)
    comando = [PY, '/content/TripoSR/run.py', str(imagen), '--output-dir', SALIDA, '--model-save-format', 'obj', '--mc-resolution', str(res), '--chunk-size', '4096']
    r = subprocess.run(comando)
    if r.returncode == 0:
        break
    print(f'Falló con código {r.returncode} (probable falta de memoria). Reintentando con menor resolución...')
else:
    raise RuntimeError('TripoSR no pudo generar la malla ni con la resolución más baja. Activa una GPU o reinicia el entorno de ejecución.')
archivos = [p for p in pathlib.Path(SALIDA).rglob('*') if p.is_file()]
if not archivos:
    raise FileNotFoundError('No se encontraron archivos de salida.')
for archivo in archivos:
    print(archivo, f'({archivo.stat().st_size / 1024:.1f} KB)')


## 4. Pulir la malla (opcional)
Elimina fragmentos sueltos, repara huecos pequeños y suaviza la superficie. Genera `figura_pulida.obj`. Si el suavizado borra demasiado detalle, baja `iterations` o sáltate esta celda.

In [ ]:
import trimesh, pathlib
SALIDA = pathlib.Path('/content/3dst-output')
PULIDO = SALIDA / 'pulido'
PULIDO.mkdir(exist_ok=True)
origen = sorted(SALIDA.rglob('mesh.obj'))
if not origen:
    raise FileNotFoundError('No se encontró mesh.obj. Ejecuta antes la celda de generación.')
malla = trimesh.load(str(origen[0]), force='mesh')
print('Original:', len(malla.vertices), 'vértices,', len(malla.faces), 'caras')
# 1) Quitar fragmentos sueltos: conservar solo la pieza más grande
partes = malla.split(only_watertight=False)
if len(partes) > 1:
    malla = max(partes, key=lambda m: len(m.faces))
# 2) Reparar huecos pequeños y normales
trimesh.repair.fill_holes(malla)
trimesh.repair.fix_normals(malla)
# 3) Suavizado Taubin: alisa la superficie sin encoger la figura
trimesh.smoothing.filter_taubin(malla, lamb=0.5, nu=-0.53, iterations=10)
malla.export(str(PULIDO / 'figura_pulida.obj'))
print('Pulida:', len(malla.vertices), 'vértices,', len(malla.faces), 'caras')
print('Guardada en', PULIDO)

## 5. Descargar el modelo 3D

In [ ]:
from google.colab import files
import pathlib
salida = pathlib.Path('/content/3dst-output')
modelos = sorted((salida / 'pulido').glob('*.obj')) or sorted(salida.rglob('*.obj'))
if not modelos:
    raise FileNotFoundError('No se encontró ningún archivo OBJ.')
for modelo in modelos:
    print('Descargando:', modelo.name)
    files.download(str(modelo))

**Importante:** Colab puede no ofrecer GPU en todas las sesiones. El modelo es una estimación visual, no una reproducción dimensional exacta.